# QPoland — Quantum-Ready Production Scheduling
## MetalTech ERP Challenge | Google Colab

**Goal:** minimize makespan for jobs that must pass through machines in the fixed order **T → F → S** (lathe → milling → welding), while ensuring each operation runs once, machine capacity is respected, and job precedence is preserved.

This notebook delivers both required baseline instances:
- **Instance A:** 2 jobs × 2 machines (T → F), exported as `schedule_2x2.json`
- **Instance B:** 3 jobs × 3 machines (T → F → S), exported as `schedule_3x3.json`

It also provides an extensible QUBO construction for one-hot start-time variables and a transparent solver comparison. For these tiny instances, the deterministic exact enumerator is the correctness oracle; the simulated annealing backend is a heuristic. An optional Qiskit path is left as an extension point. **No quantum advantage is claimed.**


## 1. Install optional solver packages
Run this cell in Google Colab. The baseline only needs the Python standard library; OR-Tools and dimod are optional comparisons.

In [ ]:
%pip -q install ortools dimod dwave-neal


## 2. Define the challenge instances

In [ ]:
import itertools, json, os, time, random
from pathlib import Path

OUT = Path('/content/qpoland_results')
OUT.mkdir(exist_ok=True)

# Processing durations are challenge-demo data; replace with organizer-provided durations if supplied.
INSTANCE_A = {
    'name': 'instance_a_2x2',
    'machines': ['T', 'F'],
    'jobs': {
        'J1': {'T': 2, 'F': 1},
        'J2': {'T': 1, 'F': 2},
    }
}
INSTANCE_B = {
    'name': 'instance_b_3x3',
    'machines': ['T', 'F', 'S'],
    'jobs': {
        'Z1': {'T': 2, 'F': 1, 'S': 2},
        'Z2': {'T': 1, 'F': 2, 'S': 1},
        'Z3': {'T': 2, 'F': 2, 'S': 1},
    }
}
print('Instances ready:', INSTANCE_A['name'], INSTANCE_B['name'])


## 3. Schedule evaluator and exact reference solver

In [ ]:
def evaluate_order(instance, order):
    # Permutation flow-shop schedule: every machine processes jobs in the same order.
    machines = instance['machines']
    jobs = instance['jobs']
    end_on_machine = {m: 0 for m in machines}
    job_end = {j: 0 for j in jobs}
    operations = []
    for j in order:
        prev_end = 0
        for m in machines:
            start = max(prev_end, end_on_machine[m])
            duration = jobs[j][m]
            finish = start + duration
            operations.append({'job': j, 'machine': m, 'start': start,
                               'duration': duration, 'finish': finish})
            end_on_machine[m] = finish
            prev_end = finish
        job_end[j] = prev_end
    return max(job_end.values()), operations

def solve_exact(instance):
    names = list(instance['jobs'])
    best = None
    for order in itertools.permutations(names):
        makespan, ops = evaluate_order(instance, order)
        candidate = {'makespan': makespan, 'order': list(order), 'operations': ops}
        if best is None or (candidate['makespan'], candidate['order']) < (best['makespan'], best['order']):
            best = candidate
    return best

for inst in (INSTANCE_A, INSTANCE_B):
    result = solve_exact(inst)
    print(inst['name'], 'best order =', result['order'], '| makespan =', result['makespan'])


## 4. Validate all hard constraints

In [ ]:
def validate_schedule(instance, result):
    ops = result['operations']
    # Every required operation appears exactly once
    expected = {(j, m) for j, spec in instance['jobs'].items() for m in instance['machines']}
    observed = [(o['job'], o['machine']) for o in ops]
    assert len(observed) == len(set(observed)) and set(observed) == expected
    # No overlap on a machine
    for m in instance['machines']:
        machine_ops = sorted([o for o in ops if o['machine'] == m], key=lambda x: x['start'])
        for a, b in zip(machine_ops, machine_ops[1:]):
            assert a['finish'] <= b['start'], (m, a, b)
    # Precedence follows machine order for every job
    rank = {m:i for i,m in enumerate(instance['machines'])}
    for j in instance['jobs']:
        job_ops = sorted([o for o in ops if o['job'] == j], key=lambda x: rank[x['machine']])
        for a,b in zip(job_ops, job_ops[1:]):
            assert a['finish'] <= b['start'], (j, a, b)
    assert max(o['finish'] for o in ops) == result['makespan']
    return {'feasible': True, 'makespan': result['makespan'], 'operations': len(ops)}

for inst in (INSTANCE_A, INSTANCE_B):
    sol = solve_exact(inst)
    print(inst['name'], validate_schedule(inst, sol))


## 5. Export the two mandatory JSON schedules

In [ ]:
def export_challenge_json(instance, result, filename):
    # Explicit and easy to audit. Adapt field names if the challenge submission schema differs.
    payload = {
        'instance': instance['name'],
        'objective': 'minimize_makespan',
        'makespan': result['makespan'],
        'job_order': result['order'],
        'feasible': True,
        'machines': instance['machines'],
        'operations': sorted(result['operations'], key=lambda o: (o['start'], o['machine'], o['job']))
    }
    path = OUT / filename
    path.write_text(json.dumps(payload, indent=2))
    print('Wrote', path)
    return payload

sol_a = solve_exact(INSTANCE_A)
sol_b = solve_exact(INSTANCE_B)
json_a = export_challenge_json(INSTANCE_A, sol_a, 'schedule_2x2.json')
json_b = export_challenge_json(INSTANCE_B, sol_b, 'schedule_3x3.json')
print(json.dumps(json_b, indent=2))


## 6. Optional simulated annealing baseline (dimod / neal)

In [ ]:
def run_annealing_if_available(instance, num_reads=200):
    try:
        import dimod
        import neal
    except Exception as e:
        print('Optional annealing packages unavailable:', e)
        return None
    # Small assignment QUBO: one binary variable y[j,p] assigns job j to sequence position p.
    # Penalties enforce exactly one job per position and one position per job.
    # A pairwise sequence-dependent cost term can be added for general instances.
    jobs = list(instance['jobs'])
    n = len(jobs)
    bqm = dimod.BinaryQuadraticModel({}, {}, 0.0, dimod.BINARY)
    x = {(j,p): f'x_{j}_{p}' for j in range(n) for p in range(n)}
    A = 10.0
    for j in range(n):
        bqm.add_linear(x[j,0], 0.0)  # initialize all variables
    for j in range(n):
        terms = [x[j,p] for p in range(n)]
        bqm.offset += A
        for v in terms: bqm.add_variable(v, -A)
        for a in range(n):
            for b in range(a+1,n): bqm.add_interaction(terms[a], terms[b], 2*A)
    for p in range(n):
        terms = [x[j,p] for j in range(n)]
        bqm.offset += A
        for v in terms: bqm.add_variable(v, -A)
        for a in range(n):
            for b in range(a+1,n): bqm.add_interaction(terms[a], terms[b], 2*A)
    samples = neal.SimulatedAnnealingSampler().sample(bqm, num_reads=num_reads)
    for sample in samples.samples():
        order_idx = []
        valid = True
        for p in range(n):
            chosen = [j for j in range(n) if sample[x[j,p]] == 1]
            if len(chosen) != 1: valid = False; break
            order_idx.append(chosen[0])
        if valid and len(set(order_idx)) == n:
            order = [jobs[j] for j in order_idx]
            ms, ops = evaluate_order(instance, order)
            return {'order': order, 'makespan': ms, 'operations': ops, 'method': 'simulated_annealing'}
    return {'method':'simulated_annealing', 'feasible':False}

for inst in (INSTANCE_A, INSTANCE_B):
    ann = run_annealing_if_available(inst)
    print(inst['name'], ann if ann is not None else 'not run')


## 7. QUBO formulation notes and scaling

For an operation-start-time formulation, define binary variables \(x_{j,m,t}=1\) when operation \((j,m)\) starts at integer time \(t\). A standard QUBO combines:
- **Exactly one start:** \(A(1-\sum_t x_{j,m,t})^2\)
- **Precedence:** penalize start choices that place the next operation before the previous operation finishes
- **Machine non-overlap:** penalize pairs of operations whose chosen intervals overlap on the same machine
- **Makespan:** minimize the maximum completion time, commonly via an auxiliary horizon/indicator encoding or by solving a sequence of feasibility horizons.

This encoding is transparent but can grow quickly with horizon and operation count. The assignment QUBO in the optional cell is a compact sequencing subproblem, not a full proof that the complete time-indexed scheduling problem has been encoded. For a production-grade extension, compare against OR-Tools CP-SAT and report feasibility, makespan, runtime, and optimality gap. The QUBO/annealing results are heuristic unless optimality is independently established.


## 8. Optional OR-Tools CP-SAT comparison

In [ ]:
def run_ortools(instance, time_limit=10):
    try:
        from ortools.sat.python import cp_model
    except Exception as e:
        print('OR-Tools unavailable:', e)
        return None
    model = cp_model.CpModel()
    jobs = list(instance['jobs'])
    machines = instance['machines']
    horizon = sum(sum(spec.values()) for spec in instance['jobs'].values())
    starts, ends, intervals = {}, {}, {}
    machine_intervals = {m: [] for m in machines}
    for j in jobs:
        for mi,m in enumerate(machines):
            d = instance['jobs'][j][m]
            s = model.NewIntVar(0, horizon, f's_{j}_{m}')
            e = model.NewIntVar(0, horizon, f'e_{j}_{m}')
            iv = model.NewIntervalVar(s, d, e, f'i_{j}_{m}')
            starts[j,m], ends[j,m], intervals[j,m] = s,e,iv
            machine_intervals[m].append(iv)
            if mi:
                prev = machines[mi-1]
                model.Add(starts[j,m] >= ends[j,prev])
    for m in machines:
        model.AddNoOverlap(machine_intervals[m])
    makespan = model.NewIntVar(0, horizon, 'makespan')
    model.AddMaxEquality(makespan, [ends[j,machines[-1]] for j in jobs])
    model.Minimize(makespan)
    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = time_limit
    status = solver.Solve(model)
    if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        return {'status': solver.StatusName(status)}
    ops=[]
    for j in jobs:
        for m in machines:
            ops.append({'job':j,'machine':m,'start':solver.Value(starts[j,m]),
                        'duration':instance['jobs'][j][m],'finish':solver.Value(ends[j,m])})
    return {'status':solver.StatusName(status),'makespan':solver.Value(makespan),'operations':ops}

for inst in (INSTANCE_A, INSTANCE_B):
    print(inst['name'], run_ortools(inst))


## 9. Download results

In [ ]:
from google.colab import files
files.download(str(OUT / 'schedule_2x2.json'))
files.download(str(OUT / 'schedule_3x3.json'))


## Submission notes
- Both JSON files are generated by the notebook and validated against operation uniqueness, precedence, machine non-overlap, and makespan consistency.
- Processing durations above are clearly labelled demo values because the screenshots do not specify a numeric processing-time table. Replace them with the organizer's exact instance data if available.
- Instance A uses T → F; Instance B uses T → F → S.
- This baseline uses an exact enumerator for the tiny required instances. The notebook includes optional simulated annealing and CP-SAT comparison cells. A real QPU backend can be added once the event specifies provider credentials and target hardware.
